# UPP 1k-step decision: A vs FX vs L0 (2x T4)

Follow-up to `soro-upp-p0-smoke`. 100 steps was enough to kill F0 and keep A. FX was barely above chance.
This run is 1000 steps, 4000 rows, batch 4, Qwen3-0.6B-Base.

| ID | What |
| A | Kev pointer (query first, options see the request) |
| FX | block layout + option-to-request xattn |
| L0 | block layout + 4 gists + ColBERT MaxSim (late interaction) |

GPU0: A then L0. GPU1: FX. No ToolRet, no Gemma, no refiner. Stop if any config OOMs.

In [ ]:
import os, subprocess, sys, time, pathlib, json
REPO_URL = "https://github.com/nisaral/Soro-Boro.git"
BRANCH = "upp-p0"
WORK = "/kaggle/working" if os.path.isdir("/kaggle") else "/content"
REPO = f"{WORK}/Soro-Boro"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "fetch", "--depth", "1", "origin", BRANCH], check=False)
    subprocess.run(["git", "-C", REPO, "checkout", BRANCH], check=False)
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only", "origin", BRANCH], check=False)
os.chdir(REPO)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout, flush=True)

In [ ]:
import torch
!pip -q install -U "transformers>=5.0" peft bitsandbytes accelerate huggingface_hub safetensors
!nvidia-smi --query-gpu=name,memory.total --format=csv
print(torch.__version__, torch.cuda.device_count(), "GPU(s)", flush=True)
assert torch.cuda.device_count() >= 1, "need a GPU"

In [ ]:
COMMON = (
    "--base-model Qwen/Qwen3-0.6B-Base --train-rows 4000 --eval-rows 120 --batch 4 "
    "--max-seq-len 832 --max-steps 1000 --log-every 50"
)
t0 = time.time()
gpu0 = subprocess.Popen(
    f"CUDA_VISIBLE_DEVICES=0 python veyra/training/p0_upp_ablation.py --configs A,L0 {COMMON} --out runs/p0_1k0 > runs_p0_1k0.log 2>&1",
    shell=True,
)
gpu1 = subprocess.Popen(
    f"CUDA_VISIBLE_DEVICES=1 python veyra/training/p0_upp_ablation.py --configs FX {COMMON} --out runs/p0_1k1 > runs_p0_1k1.log 2>&1",
    shell=True,
)
print("started A,L0 on GPU0 and FX on GPU1", flush=True)
gpu0.wait(); gpu1.wait()
print(f"done in {(time.time()-t0)/60:.1f} min  rc={gpu0.returncode},{gpu1.returncode}", flush=True)
print(open("runs_p0_1k0.log").read()[-4000:])
print("---")
print(open("runs_p0_1k1.log").read()[-4000:])

In [ ]:
res = {}
for d in ("runs/p0_1k0", "runs/p0_1k1"):
    p = pathlib.Path(d) / "results.json"
    if p.exists():
        res.update(json.loads(p.read_text()))
pathlib.Path("runs").mkdir(exist_ok=True)
pathlib.Path("runs/p0_1k.json").write_text(json.dumps(res, indent=2))
sys.path.insert(0, "veyra/training")
from p0_upp_ablation import markdown
print(markdown(dict(sorted(res.items()))))
chance = 0.22
for k, v in sorted(res.items()):
    sel = (v.get("eval") or {}).get("selection", {}).get("acc")
    zs = (v.get("eval") or {}).get("zeroshot", {}).get("acc")
    print(f"GATE {k} selection={sel} zeroshot={zs} above_chance={sel is not None and sel > chance}")